# Сборный Проект — 1

**Описание проекта**

Интернет-магазин «Стримчик» продаёт по всему миру компьютерные игры. Из открытых источников доступны исторические данные о продажах игр, оценки пользователей и экспертов, жанры и платформы. Нужно выявить определяющие успешность игры закономерности. Это позволит сделать ставку на потенциально популярный продукт и спланировать рекламные кампании. Данные до 2016 года, планируется кампания на 2017-й. Нужно отработать принцип работы с данными.

В наборе данных попадается аббревиатура *ESRB (Entertainment Software Rating Board)* — это ассоциация, определяющая возрастной рейтинг компьютерных игр. *ESRB* оценивает игровой контент и присваивает ему подходящую возрастную категорию.

**Цель проекта** - выявить определяющие успешность игры закономерности,что позволит сделать ставку на потенциально популярный продукт и спланировать рекламные кампании.

**Описание данных**

- *Name* — название игры
- *Platform* — платформа
- *Year_of_Release* — год выпуска
- *Genre* — жанр игры
- *NA_sales —* продажи в Северной Америке (миллионы проданных копий)
- *EU_sales* — продажи в Европе (миллионы проданных копий)
- *JP_sales* — продажи в Японии (миллионы проданных копий)
- *Other_sales —* продажи в других странах (миллионы проданных копий)
- *Critic_Score* — оценка критиков (максимум 100)
- *User_Score* — оценка пользователей (максимум 10)
- *Rating* — рейтинг от организации *ESRB* (англ. *Entertainment Software Rating Board).* 

**План работы**

1. Предобработка данных
    * замена названия столбцов
    * изменение типа данных
    * обработка пропущенных значений
    * посчет суммарных продаж во всех регионах


2. Исследовательский анализ данных
    * сколько игр выпускалось в разные годы
    * как менялись продажи по платформам
    * какие платформы лидируют по продажам, растут или падают
    * как влияют на продажи внутри одной популярной платформы отзывы пользователей и критиков
    * распределение игр по жанрам
    
    
3. Портрет пользователя каждого региона
    * самые популярные платформы (топ-5)
    * самые популярные жанры (топ-5)
    * влияет ли рейтинг ESRB на продажи в отдельном регионе
    
    
4. Проверка гипотез
5. Вывод

## Откройте файл с данными и изучите общую информацию

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from scipy import stats as st

In [ ]:
data = pd.read_csv('\\datasets\\games.csv')

In [ ]:
# выведим первые 10 строк датасета
data.head(10)

In [ ]:
# выведим основную информацию
data.info()

In [ ]:
# проверим датасет на наличие дубликатов 
data.duplicated().sum()

**Вывод**

Всего в файле 16715 записи. В датасете видны пропущенные значения в ряде столбцов, типы данных некоторых колонок не соответствуют значениям, дубликаты отсутствуют 

## Подготовьте данные

### Замените названия столбцов (приведите к нижнему регистру)

In [ ]:
data.columns = data.columns.str.lower()

In [ ]:
# проверяем результат 
data.info()

### Преобразуйте данные в нужные типы

На первый взгляд логично привести целочисленному типу (int) два столбца `year_of_release`, `critic_score`, т.к в остальных колонках данные - дробные числа. Кроме столбов  с названиями: `name`, `platform`, `genre`, `rating`. Также `user_score` необходимо привести с типу float. 
Посмотрим уникальные значения в этих столбцах 

In [ ]:
data['year_of_release'].sort_values().unique()

In [ ]:
data['critic_score'].sort_values().unique()

In [ ]:
data['user_score'].sort_values().unique()

Данные в столбцах `year_of_release` и `critic_score`, не могут быть изменены, т.к внутри колонок не все данные конвертируемы к int. На данный момент оставляем как есть. 
В `user_score` есть текстовое значение 'tbd', означает "to be defined" - предстоит определить, то есть на данный момент это пропуск в данных, заменим его на 'nan' и приведем к типу  float

In [ ]:
data['user_score'] = data['user_score'].replace('tbd', np.nan)
data['user_score'] = data['user_score'].astype(float)

### Обработайте пропуски при необходимости

In [ ]:
# проверим количество пропусков в датасете
data.isna().sum()

In [ ]:
# смотрим колличество пропусков в %  
pd.DataFrame(round(data.isna().mean()*100,)).style.background_gradient('coolwarm')

В столбце `critic_score` и `user_score` пропусков больше 40% и их невозможно заменить подходящим значением, оставляем их без изменений.В `rating` помечаем пропуски значением *'tbd'* (понадобится для анализа).
Далее смотрим на пропуски в столбце `name` и `genre` - их всего 2

In [ ]:
data['rating'] = data['rating'].fillna('tbd')

In [ ]:
data[data['name'].isna()]

Два пропуска в столбцах `name` и `genre` - это одни и те же строки, их можно удалить

In [ ]:
data.dropna(subset=['name'], inplace=True)

В столбеце `year_of_release` пропусков меньше 5%, можем их удалить. И зменим тип данных на целочисленный

In [ ]:
data.dropna(subset=['year_of_release'], inplace=True)
data['year_of_release'] = data['year_of_release'].astype(int)

In [ ]:
# проверяем результат
data.isna().sum()

*Причины, которые могли привести к пропускам в данных:*

Одной из частых причин таких ошибок — человеческий фактор, но также бывают и технические причины. Пропуски в данных могут появиться разными путями. Возможно ситема сбора информации дала сбой или данные еще не обнародованы, также пропуски оставляют умышленно, рассчитывая на автозаполнение спецсимволами.

### Посчитайте суммарные продажи во всех регионах

In [ ]:
data['total_sales'] = data[['na_sales', 'eu_sales', 'jp_sales', 'other_sales']].sum(axis='columns')

In [ ]:
data.head()

**Вывод**

Данные были преобразованы для дальнейшего использования. Данные были приведены к корректным типам, были обработаны пропуски, в некоторых случаях строки были удалены, в одном столбце пропуски были заменены на значение 'tbd', также были расчитаны суммарные продажи по регионам и записаны в отдельный столбец 'total_sales'

In [ ]:
temp = data.copy() 
list_c = ['name', 'platform', 'year_of_release', 'genre', 'critic_score', 'user_score', 'rating']
print(temp.info())
for col_l in list_c:
  print('-'* 25)
  print(col_l, temp[col_l].sort_values().unique())
  print(col_l,': кол-во NaN',temp[col_l].isna().sum(),
        ', процент NaN', round(temp[col_l].isna().sum()/len(temp)*100, 2),'%')

## Проведите исследовательский анализ данных

### Сколько игр выпускалось в разные годы

In [ ]:
(
    data.pivot_table(index='year_of_release', values='total_sales', aggfunc='count')
    .plot(grid=True, figsize=(10, 6), title = 'Кол-во выпущенных игр за все годы', 
          ylabel='Кол-во игр', xlabel='Год релиза')
)
plt.show()

**Важны ли данные за все периоды?**

Из графика видно, что рост кол-ва игр начался с 2000 года, наблюдается застой с 2002 по 2004 года, резкий скачок до 2008 год, где виден пик по кол-ву выпущенных игр, затем спад до 2016. 
Делаем вывод, что актуальными являются данные за прошедшие 10 лет, которые отражают как взлет, так и падение количества игр

### Как менялись продажи по платформам

**Выберите платформы с наибольшими суммарными продажами и постройте распределение по годам**

In [ ]:
total_sales_per_platform = (data.groupby(['platform'],as_index = False)['total_sales'].sum()
                                 .sort_values('total_sales',ascending = False)
                           )

total_sales_per_platform.plot(grid=True, kind='bar', x='platform', y='total_sales', 
                              figsize=(10, 5), title = 'Суммарные продажи по платформам',
                             xlabel='Платформа', ylabel='Продажи')
plt.show()

In [ ]:
top_6 = total_sales_per_platform.head(6)['platform'].tolist()
top_6

На графике выделяется 6 платформ с наибольшими суммарными продажами. Это: PS2, X360, PS3, Wii, DS, PS. Смотрим, как менялись продажи по этим платформам и построим распределение по годам. 

In [ ]:
for name in top_6:
    data.query('platform == @name').pivot_table(index = 'year_of_release', 
                values = 'total_sales', aggfunc = 'sum').sort_values('year_of_release',
                                    ascending = False)['total_sales'].plot(figsize = (10, 5), 
                                    grid=True, label=name)
                                                                    
    plt.legend()
    plt.xlabel('Дата релиза', labelpad = 10)
    plt.ylabel('Продажи', labelpad = 10)
    plt.title('Распределение по годам')

In [ ]:
data.query('platform == "DS" & year_of_release < 2004')

In [ ]:
data = data.drop(data[(data.platform == 'DS') & (data.year_of_release == 1985)].index)

In [ ]:
for name in top_6:
    data.query('platform == @name').pivot_table(index = 'year_of_release', 
                values = 'total_sales', aggfunc = 'sum').sort_values('year_of_release',
                                        ascending = False)['total_sales'].plot(figsize = (10, 5), 
                                                                               grid=True, label=name)
                                                                    
    plt.legend()
    plt.xlabel('Дата релиза', labelpad = 10)
    plt.ylabel('Продажи', labelpad = 10)
    plt.title('Распределение по годам')

**За какой характерный срок появляются новые и исчезают старые платформы?**

Оценим продолжительность жизни платформ

In [ ]:
# вычислим разность между годом окончания существования платформы и началом
platform_lifetime = data.pivot_table(index='platform', values='year_of_release', aggfunc=[min, max])
platform_lifetime.columns = ['beginnig', 'end']
platform_lifetime['duration'] = platform_lifetime['end'] - platform_lifetime['beginnig']
# посмотрим результат
display(platform_lifetime['duration'].describe())
display(platform_lifetime[platform_lifetime['duration'] == 31])

Средний срок жизни платформы составляет 7 лет, медиана - почти 8. Платформа-долгожитель - PC (персональный компьютер), 32 года. 
По графику же видно, что продолжительность жизни топ 6 платформ составляет примерно 10 лет.

**Возьмите данные за соответствующий актуальный период. Актуальный период определите самостоятельно в результате исследования предыдущих вопросов. Основной фактор — эти данные помогут построить прогноз на 2017 год. Не учитывайте в работе данные за предыдущие годы**

Для формирования прогноза на 2017 год имеет смысл рассмотреть период за последние 4 года, т.е будем опираться на данные с 2013 года, как наиболее актуальные(год, когда платформы еще достаточно популярны, а устаревшие ушли с рынка)
Отрезаем данные до 2013 года и сохраним их в таблице actual_platforms. 

In [ ]:
data_actual = data.query('year_of_release >= 2013').copy()

### Какие платформы лидируют по продажам, растут или падают

**Выберите несколько потенциально прибыльных платформ**

In [ ]:
top_actual_platforms = (data_actual.groupby(['platform'],as_index = False)['total_sales'].sum()
                        .sort_values('total_sales',ascending = False)
                       )

(top_actual_platforms.plot(kind='bar',x='platform', y='total_sales',figsize=(10, 5), grid=True)
                         .set(xlabel='Платформа', ylabel='Продажи')
)
plt.show()

In [ ]:
top_platforms = top_actual_platforms['platform'].tolist()
top_platforms 

In [ ]:
for name in top_platforms:
    data_actual.query('platform == @name').pivot_table(index = 'year_of_release',
                    values = 'total_sales', aggfunc = 'sum').sort_values('year_of_release',
                    ascending = False)['total_sales'].plot(figsize=(10, 5), grid=True, label = name)
    
    plt.xticks(np.arange(2013, 2017, step=1))
    plt.xlabel('Дата релиза', labelpad = 10)
    plt.ylabel('Продажи', labelpad = 10)
    plt.legend()

Продажи на всех платформах падают. На платформе PSP продажи в 2016г вовсе прекратились, минимальные продажи зафиксированы у платформ X360, Wii,DS, поэтому данные в настоящее время не актуальны.
Лучший результат по сравнению с другими у PS4 и XOne

**Постройте график «ящик с усами» по глобальным продажам игр в разбивке по платформам**

In [ ]:
# полный график
sns.set(rc={'figure.figsize':(10,5)}) #настраиваем размер графика
sns.boxplot(x="platform", y="total_sales", data=data_actual, palette="Spectral")

In [ ]:
# масштабированный 
sns.set(rc={'figure.figsize':(10,5)}) #настраиваем размер графика
ax = sns.boxplot(x="platform", y="total_sales", data=data_actual, palette="Spectral")
ax.set(ylim=(0,5))

In [ ]:
data_actual.pivot_table(index = 'platform',values='total_sales',aggfunc = ['mean','median']) 

Практически по всем платформам существуют выбросы. 
У платформ PS4, X360, XOne приблизительно равны медианные значения продаж (0.20, 0.26, 0.22), а также среднее арифметическое значение продаж (0.80,0.73, 0.64). Можно предположить, что большей популярностью пользуются именно эти платформы

### Как влияют на продажи внутри одной популярной платформы отзывы пользователей и критиков
**Постройте диаграмму рассеяния и посчитайте корреляцию между отзывами и продажами**

In [ ]:
ps4 = data_actual.query("platform == 'PS4'")
ps4[ps4['critic_score'].isna() == False].plot(x='critic_score', y='total_sales', 
                            title='оценка критиков', kind='scatter',figsize=(10, 5), color='b')

ps4[ps4['user_score'].isna() == False].plot(x='user_score', y='total_sales', 
                        title='оценка пользователей',kind='scatter',figsize=(10, 5), color='green')
plt.show()

# Расчет коэффициента корреляции
print(f'Корреляция между оценкой критиков и продажами: {ps4.critic_score.corr(ps4.total_sales):.2}')
print(f'Корреляция между оценкой пользователей и продажами: {ps4.user_score.corr(ps4.total_sales):.2}')

Оценка критиков для платформы PS4 имеет слабое влияние на продажи (коэфф. 0,4), оценка пользователей на продажи не влияет

**Соотнесите выводы с продажами игр на других платформах**

In [ ]:
# используем цикл для построения диаграмм рассеяния 
for name in top_platforms:
    x_critic = data_actual[data_actual['platform']==name]['critic_score']
    y_critic = data_actual[data_actual['platform']==name]['total_sales']
    
    x_user = data_actual[data_actual['platform']==name]['user_score']
    y_user = data_actual[data_actual['platform']==name]['total_sales']
    
    fig = plt.figure(figsize=(13.5, 4.5))
    
    ax1 = fig.add_subplot(121)
    ax1.scatter(x_critic, y_critic, c='b')
    ax1.set_title('Зависимость продаж от оценок критиков\n на платформе '+name, 
                  fontsize=14, fontweight="bold")
    ax1.set_xlabel('Оценки критиков\n', fontsize=14)
    ax1.set_ylabel('Продажи', fontsize=14)
    
    ax2 = fig.add_subplot(122)
    ax2.scatter(x_user, y_user, c='green')
    ax2.set_title('Зависимость продаж от оценок пользователей\n на платформе '+name, 
                  fontsize=14, fontweight="bold")
    ax2.set_xlabel('Оценки пользователей\n', fontsize=14)
    ax2.set_ylabel('Продажи', fontsize=14)
    
    plt.tight_layout(w_pad=3)

In [ ]:
# считаем коэффициент корреляции
for name in top_platforms:
    print(name)
    data_select = data.query('platform == @name')
    print(f'Корреляция между оценкой критиков и продажами: {data_select.critic_score.corr(data_select.total_sales):.2}')
    print(f'Корреляция между оценкой пользователей и продажами: {data_select.user_score.corr(data_select.total_sales):.2}')
    print()

**Вывод**

Значения корреляций отличаются, но общий тренд есть. По диаграммам рассеяния видим, что чем выше оценки критиков, тем выше продажи игр на платформах. 
<br>
Оценка критиков немного сильнее коррелирует с продажами, хотя эта связь слабая. Корреляция оценок пользователей и продаж очень слабая. 
<br>
В целом, при прогнозе продаж немного надежнее полагаться на отзывы критиков, но мы не можем утверждать, что высокие оценки критиков приводят к высоким продажам игр, т.к корреляция не говорит о причинно-следственной связи

### Общее распределение игр по жанрам
**Что можно сказать о самых прибыльных жанрах? Выделяются ли жанры с высокими и низкими продажами?**

In [ ]:
# график с медианными продажами по жанрам 
(data_actual
    .pivot_table(index='genre', values='total_sales', aggfunc='median')
    .sort_values('total_sales', ascending=False)
    .plot(kind='bar', y='total_sales', figsize=(10, 5), 
          legend=False,title = 'Распределение по жанрам')
    .set(xlabel='Жанр', ylabel='Продажи'))

plt.show()

In [ ]:
# диаграмма размаха для всех жанров
(data_actual[['total_sales', 'genre']].boxplot(column='total_sales', by='genre', figsize=(10,8))
)
plt.xlabel('Жанр')
plt.ylabel('Количество проданных копий, млн')
plt.title('Диаграммы размаха по продажам в разных жанрах')
plt.show()

In [ ]:
# диаграмма размаха в масштабе 
(data_actual[['total_sales', 'genre']].boxplot(column='total_sales', by='genre', figsize=(15,8))
)
plt.xlabel('Жанр')
plt.ylabel('Количество проданных копий, млн')
plt.title('Диаграммы размаха по продажам в разных жанрах')
plt.ylim(0, 4)
plt.show()

**Вывод**

В результате исследовательского анализа делаем следующие выводы: 

- рост кол-ва игр начался с 2000 года, пик пришелся на 2008 год, затем наблюдается спад
- платформы с наибольшими суммарными продажами за все время: PS2, X360, PS3, Wii, DS, PS.
- характерный срок жизни платформы составляет от 8 до 10 лет, платформа-долгожитель - PC (32 года)
- актуальный период для прогноза на 2017 год - период за последние 4 года (данные с 2013 года)
- потенциально прибыльные платформы: PS4, X360, XOne, WiiU, 3DS
- взаимосвязи между отзывами пользователей и продажами нет, между отзывами критиков и продажами существует, однако слабая
- потенциально наличие устойчивого спроса пользователей на жанры: шутеры, спортивные игры, платформеры и RPG.

## Составьте портрет пользователя каждого региона (NA, EU, JP)
**Определите для пользователя каждого региона (*NA, EU, JP*):**

- Самые популярные платформы (топ-5). Опишите различия в долях продаж.
- Самые популярные жанры (топ-5). Поясните разницу.
- Влияет ли рейтинг ESRB на продажи в отдельном регионе?

### Самые популярные платформы (топ-5)
**Опишите различия в долях продаж**

In [ ]:
regions_list = ['na_sales','eu_sales','jp_sales']
data_actual[regions_list].sum()

In [ ]:
for region in regions_list:
    (data_actual.pivot_table(index='platform', values=region, aggfunc='sum')
                .sort_values(region, ascending=False).head(5)
                .plot(kind='pie', 
                      y=region, 
                      autopct='%1.0f%%', 
                      figsize=(6, 6), 
                      legend=False, 
                      title = region)
                .set(ylabel='Продажи')
    )
    plt.show()

В Европе и Америке лидируют PS и Xbox. В Японии с 50% рынка занимает их портативная консоль 3DS.

In [ ]:
for region in regions_list:
    (data_actual.pivot_table(index='genre', values=region, aggfunc='sum')
                .sort_values(region, ascending=False).head(5)
                .plot(kind='pie', 
                      y=region, 
                      autopct='%1.0f%%', 
                      figsize=(6, 6), 
                      legend=False, 
                      title = region)
                .set(ylabel=None)
    )
    plt.show()

### Самые популярные жанры (топ-5)

В Европе и Америке популярны жанры Action, Shooter и Sports. В Япония любят РПГ и Action.

In [ ]:
games_fresh = data_actual.copy()
countries = games_fresh.columns[4:7]
# начертим круговую диаграмму для каждого региона
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('ТОП-5 жанров в регионах')
for i in range(3):
  # выберем 5 самых продающих жанров
  df_i = (games_fresh.groupby('genre')[countries[i]].sum().
             reset_index().
             sort_values(countries[i], ascending = False, ignore_index = True).
             head(5))
  # занесем все остальыне продажи в другие жанры
  df_i = df_i.append({'genre': 'Other_genre', countries[i]: games_fresh[countries[i]].sum() - df_i[countries[i]].sum()}, ignore_index= True)
  axs[i].pie(df_i[countries[i]], labels = df_i['genre'], autopct='%1.1f%%')
  # Зададим название региона.
  axs[i].set_title(df_i.columns[1].split('_')[0].upper())
plt.show()

### Влияет ли рейтинг ESRB на продажи в отдельном регионе

In [ ]:
for region in regions_list:
    (data_actual.pivot_table(index='rating', values=region, aggfunc='sum')
                .sort_values(region, ascending=False).head(5)
                .plot(kind='pie', 
                      y=region, 
                      autopct='%1.0f%%', 
                      figsize=(6, 6), 
                      legend=False, 
                      title = region)
                .set(ylabel='Продажи')
    )
    plt.show()

В Европе и Америке хорошо продаются игры с рейтингом для лиц старше 17 лет (М) и для всех (Е).
В Японии много продаж по играм с неизвестным рейтингом и рейтингом для всех.

**Вывод**

Портреты пользователей из Европы и Америки схожи.
Лидируют платформы PS и Xbox, жанры Action,Shooter и Sports, игры с рейтингом для взрослых и для всех.

У пользователей из Японии свои предпочтения.
Лидирует консоль 3DS, жанры РПГ и Action, игры с неизвестным рейтингом и рейтингом для всех.

*Так как рейтинг ESRB предназначен для рецензирования игр на территории Северной Америки, заметнее его влияние в этом регионе и Европе. В Японии популярны игры вовсе без этого рейтинга, скорее всего это игры, созданные в Японии и не получавшие этого рейтинга.*

## Проверьте гипотезы

- Средние пользовательские рейтинги платформ *Xbox One* и *PC* одинаковые;
- Средние пользовательские рейтинги жанров *Action* (англ. «действие», экшен-игры) и *Sports* (англ. «спортивные соревнования») разные.

### Средние пользовательские рейтинги платформ Xbox One и PC одинаковые

Сформулируем нулевую и альтернативную гипотезы:

    H_0: Средние пользовательские рейтинги платформ Xbox One и PC одинаковые
    H_1: Средние пользовательские рейтинги платформ Xbox One и PC различаются

Значение критического уровня вероятности (альфа) примем равным 5%
<br>
Пропущенные значечния отфильтруем

In [ ]:
alpha = .05 
results = st.ttest_ind(
    data_actual[(data_actual.platform == "XOne") & (data_actual.user_score.notna())]['user_score'], 
    data_actual[(data_actual.platform == "PC") & (data_actual.user_score.notna())]['user_score']
)

print('p-значение:', results.pvalue)

if results.pvalue < alpha:
    print('Отвергаем нулевую гипотезу')
else:
    print('Не получилось отвергнуть нулевую гипотезу') 

In [ ]:
(
    data_actual
    .query('platform in ["XOne", "PC"]')
    .pivot_table(index='platform', values='user_score', 
                 aggfunc=['mean', 'var'])
)

Принимаем нулевую гипотезу, p-value больше alpha, значит средний пользовательский рейтинг платформ Xbox One и PC статистически значимо не отличается. 

### Средние пользовательские рейтинги жанров Action и Sports разные

Сформулируем нулевую и альтернативную гипотезы:

    H_0: Средние пользовательские рейтинги жанров Action и Sports одинаковые
    H_1: Средние пользовательские рейтинги жанров Action и Sports различаются
    
Значение критического уровня вероятности (альфа) примем равным 5%
<br>
Пропущенные значечния отфильтруем

In [ ]:
alpha = .05 
results = st.ttest_ind(
    data_actual[(data_actual.genre == "Action") & (data_actual.user_score.notna())]['user_score'], 
    data_actual[(data_actual.genre == "Sports") & (data_actual.user_score.notna())]['user_score']
)

print('p-значение:', results.pvalue)

if results.pvalue < alpha:
    print('Отвергаем нулевую гипотезу')
else:
    print('Не получилось отвергнуть нулевую гипотезу') 

In [ ]:
(
    data_actual
    .query('genre in ["Action", "Sports"]')
    .pivot_table(index='genre', values='user_score', 
                 aggfunc=['mean', 'var'])
)

Так как принятый критический уровень вероятности 5%, нулевую гипотезу отвергаем как маловероятную. Средние пользовательские рейтинги жанров Action и Sports статистически значимо различаются

**Вывод**

Оценки пользователей не зависят от платформ, но зависят от жанров игр. На основании принятого нами значения критического уровня статистической значимости 5% можем предполагать, что первая гипотеза верна, а вторая нет.

Проверка гипотез начинается с формулировки нулевой гипотезы H₀. Исходя из H₀ формулируется альтернативная гипотеза H₁. 
<br>
H₀ всегда формулируется так, чтобы использовать знак равенства. H₁ противоречит нулевой гипотезе и принимается верным, если отбрасывается H₀. В нашем случае H₁ двусторонняя, потому что есть возможное отклонение и в большую, и в меньшую стороны. 
    
Для проверки гипотез был выбран критический уровень статистической значимости = 5%, так как это общепринятое пороговое значение 

##  Общий вывод

Входные данные были получены из открытых источников о продажах игр, оценки пользователей и экспертов, жанры и платформы. Задачей было выявить определяющие успешность игры закономерности, что позволит сделать ставку на потенциально популярный продукт и спланировать рекламные кампании в 2017 году.

Была изучена выборка данных об видеоиграх, вышедших с 1980 по 2016 годы на различных платформах - консолях и персональных компьютерах. На основе этих данных были выявлены закономерности, какие игры могут хорошо продаваться в 2017 году и составлены портреты пользователей разных регионов.

**Этап переработки данных**. На первом этапе было преведено общее изучение данных и их переработка. Так было проведено изменение регистра столбцов и типа данных. Также обработаны, удалены или заменены пропущенные значения. Посчитаны суммарные продажи во всех регионах и занесены в столбец `total_sales`.
    
**Исследовательский анализ данных**. Были сделаны следующие шаги: 
    
    - Построен график кол-ва игр выпускавшихся в разные годы
    - Построен график изменения продаж по платформам, включающий платформы с наибольшими продажами и распределением по годам.
    - По итогам анализа выбран актуальный период.
    - Построен график зависимости продаж от типа платформ за актуальный период.
    - Построен график «ящик с усами» по глобальным продажам игр в разбивке по платформам.
    - Проведен анализ влияния на продажи отзывов пользователей и критиков. Построены диаграммы рассеяния и расчитаны коэффициенты корреляции между отзывами и продажами
    - Построен график зависимости продаж от жанра за актуальный период.
    - Определены для пользователя каждого региона (*NA, EU, JP*) самые популярные платформы, жанры и выявлено влияние рейтинга ESRB на продажи в отдельном регионе. 
    - Составлены сводные таблицы и построены круговые диаграммы.

В Америке и Европе лидируют стационарные консоли - PlayStation и Xbox, среди жанров наиболее популярны  экшены и шутеры. В Японии отдельный рынок с уклоном в портативные консоли Nintendo и ролевые игры. Объём продаж в этой стране ниже, чем в Европе и Америке. 

Так как рейтинг ESRB предназначен для рецензирования игр на территории Северной Америки, заметнее его влияние в этом регионе и Европе. В Японии популярны игры вовсе без этого рейтинга, скорее всего это игры, созданные в Японии и не получавшие этого рейтинга.
 
Продажи игр сильнее коррелируют с оценкой критиков, чем с оценками пользователей. При этом оценки пользователей зависят не от платформы, а от жанра. </s>
    
    
Потенциально выгодно рекламировать игры в Европе и Америке на перспективных платформах Sony PlayStation 4 и Xbox One в жанре Shooter/Action с рейтингом для лиц старше 17 лет (М) и для всех (Е). Рынок в Японии имеет свою специфику по платформам и жанрам, кроме того не слишком обширен. В целом отзывы не влияют на продажи игр, но стоит подстраховаться и поглядывать на оценки критиков. 